# 09 — Grupimi dhe zbulimi i strukturës

        **Pyetja udhëheqëse:** Si gjejmë regjime pa etiketa të njohura?

        Ky notebook është ndërtuar si një mësim i plotë kompjuterik. Rruga kryesore ndjek:
        **problem konkret → model matematikor → zbatim nga parimet e para → bibliotekë profesionale → krahasim → interpretim**.

        ## Objektivat
        - Të implementohet k-means nga zero.
- Të kuptohet funksioni objektiv dhe inicializimi.
- Të përdoren elbow dhe silhouette.
- Të krahasohet me grupimin hierarkik.

        > **Rruga bazë për orën mësimore (45–60 min)** shënohet me seksionet **CORE**.  
        > Seksionet **EXTENDED** dhe **PLAYGROUND** janë për thellim dhe punë të pavarur.

## Pse kjo temë është e nevojshme?

Në nxënien e pambikëqyrur nuk kemi etiketa që të na tregojnë përgjigjen. Algoritmi mund të zbulojë strukturë, por edhe të imponojë strukturë sipas metrikës dhe funksionit objektiv që kemi zgjedhur. Kjo e bën grupimin veçanërisht të rëndësishëm për interpretim kritik.

k-means do të ndërtohet hap pas hapi për të parë çfarë optimizon realisht. Pastaj elbow, silhouette dhe dendrogrami do të përdoren jo si rituale, por si diagnostika që duhen lexuar bashkë me kuptimin fizik të të dhënave.

### Si duhet lexuar ky notebook

Për çdo bllok kodi do të shpjegojmë **pse** kryhet ai hap, çfarë pyetjeje i përgjigjet dhe çfarë duhet të kontrollojmë në dalje. Kodi nuk trajtohet si recetë: çdo grafik dhe çdo metrikë duhet të rikthehet te hipoteza matematikore dhe te interpretimi fizik. Kur një rezultat nuk sillet siç presim, pyetja e parë nuk është “si ta rrisim score-in?”, por “cilën hipotezë të modelit ose të të dhënave po shkelim?”.

### Para se të ekzekutojmë: Mjedisi llogaritës dhe riprodhueshmëria

Ngarkojmë bibliotekat që do të përdoren dhe fiksojmë `RANDOM_STATE = 42`. Ky seed nuk e bën problemin determinist në kuptimin fizik; ai thjesht na lejon të përsërisim të njëjtin eksperiment llogaritës dhe të diskutojmë metodën pa u hutuar nga ndryshime të rastësishme të mostrës.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Mjedisi llogaritës dhe riprodhueshmëria.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import numpy as np
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import pandas as pd
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import matplotlib.pyplot as plt
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from pathlib import Path
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import time

RANDOM_STATE = 42
rng = np.random.default_rng(RANDOM_STATE)

print("09 — Grupimi dhe zbulimi i strukturës")
print("Seed:", RANDOM_STATE)

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 1. Regjime atmosferike pa etiketa — CORE

### Para se të ekzekutojmë: Të dhëna pa etiketa

Heqim etiketat dhe ruajmë vetëm koordinatat. Kjo ndryshon natyrën e problemit: nuk kemi më një përgjigje të njohur ndaj së cilës optimizojmë accuracy; duhet të gjykojmë strukturën përmes gjeometrisë dhe interpretimit.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Të dhëna pa etiketa.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.datasets import make_blobs
# Gjenerojmë të dhëna të kontrolluara që na lejojnë të dimë strukturën që duhet të zbulojë modeli.
X,_=make_blobs(n_samples=520,centers=[[-2,-1],[0,2.3],[2.6,-0.4]],
               cluster_std=[0.6,0.75,0.65],random_state=RANDOM_STATE)
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.scatter(X[:,0],X[:,1],s=14)
ax.set_xlabel("anomalia e temperaturës"); ax.set_ylabel("anomalia e presionit")
ax.set_title("Të dhëna pa etiketa")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 2. Funksioni objektiv i k-means — CORE

k-means minimizon
$$
J=\sum_{k=1}^{K}\sum_{x_i\in C_k}\|x_i-\mu_k\|^2.
$$
Algoritmi alternon ndërmjet caktimit te qendra më e afërt dhe rillogaritjes së qendrave.

# 3. Iteracionet manuale — CORE

### Para se të ekzekutojmë: k-means hap pas hapi

Ruajmë qendrat pas çdo iteracioni që të shohim procesin, jo vetëm rezultatin final. Caktimi dhe përditësimi alternohen derisa lëvizja e qendrave të bëhet e papërfillshme.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: k-means hap pas hapi.
K=3
centers=X[rng.choice(len(X),K,replace=False)].copy()
history=[centers.copy()]
labels_history=[]
# Përsërisim procedurën për të studiuar konvergjencën ose ndjeshmërinë ndaj parametrave.
for _ in range(8):
    d2=((X[:,None,:]-centers[None,:,:])**2).sum(axis=2)
    labels=d2.argmin(axis=1)
    labels_history.append(labels.copy())
    new=np.array([X[labels==k].mean(axis=0) for k in range(K)])
    history.append(new.copy())
    if np.linalg.norm(new-centers)<1e-6: break
    centers=new

for step in [0,1,min(2,len(labels_history)-1),len(labels_history)-1]:
    # Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
    fig,ax=plt.subplots()
    lab=labels_history[min(step,len(labels_history)-1)]
    ax.scatter(X[:,0],X[:,1],c=lab,s=10)
    c=history[step]
    ax.scatter(c[:,0],c[:,1],s=130,marker="x")
    ax.set_title(f"k-means: iteracioni {step}")
    plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

### Zbatimi kanonik i k-means në `src/`

### Para se të ekzekutojmë: Kontroll me implementimin kanonik

Përdorim modulin `KMeansFromScratch` për të parë inertia-n dhe numrin e iteracioneve. Kjo lidh demonstrimin vizual me një implementim që mund të përdoret në notebook-e të tjera.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Kontroll me implementimin kanonik.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
import sys
# Kërkojmë automatikisht rrënjën e repository-t, në mënyrë që notebook-u të funksionojë nga dosje të ndryshme.
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src").is_dir():
        sys.path.insert(0, str(candidate))
        break
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from src.clustering.kmeans import KMeansFromScratch
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
km_src = KMeansFromScratch(n_clusters=3, seed=RANDOM_STATE).fit(X)
print("Inertia nga src:", round(km_src.inertia_, 3), "iteracione:", km_src.n_iter_)

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 4. Elbow dhe silhouette — CORE

### Para se të ekzekutojmë: Sa grupe duhen?

Elbow dhe silhouette japin informacione të ndryshme. Elbow kërkon një ndryshim të ritmit të uljes së inertia-s; silhouette krahason kohezionin brenda grupit me ndarjen nga grupet e tjera. Asnjëra nuk zëvendëson interpretimin e problemit.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Sa grupe duhen?.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.cluster import KMeans
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.metrics import silhouette_score
ks=range(2,8); inertia=[]; sil=[]
for k in ks:
    # Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
    m=KMeans(k,random_state=RANDOM_STATE,n_init=10).fit(X)
    inertia.append(m.inertia_)
    sil.append(silhouette_score(X,m.labels_))

# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.plot(list(ks),inertia,marker="o")
ax.set_xlabel("k"); ax.set_ylabel("Inertia")
ax.set_title("Metoda e bërrylit")
plt.show()

# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.plot(list(ks),sil,marker="o")
ax.set_xlabel("k"); ax.set_ylabel("Silhouette")
ax.set_title("Silhouette score")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

# 5. Grupimi hierarkik — CORE

### Para se të ekzekutojmë: Hierarkia e ngjashmërisë

Dendrogrami nuk kërkon të fiksojmë k që në fillim. Ai tregon rendin dhe distancat e bashkimit të grupeve, duke dhënë një pamje shumë-shkallëshe të strukturës.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Hierarkia e ngjashmërisë.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from scipy.cluster.hierarchy import linkage,dendrogram
Z=linkage(X[:80],method="ward")
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots(figsize=(10,5))
dendrogram(Z,no_labels=True,ax=ax)
ax.set_ylabel("Distanca e bashkimit")
ax.set_title("Dendrogram")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

---
## Deri këtu: materiali bazë për orën mësimore

Në një seancë rreth 60 minuta, zakonisht ndalemi këtu. Seksionet që vijojnë janë për thellim,
ushtrime të pavarura dhe eksperimentim.

# 6. Kur k-means dështon — EXTENDED

### Para se të ekzekutojmë: Një dështim i qëllimshëm i k-means

Dy gjysmëhëna kanë strukturë të qartë për syrin, por qendrat e k-means favorizojnë grupe afërsisht sferike. Ky shembull na kujton se çdo algoritëm grupimi përmban një paragjykim gjeometrik.

**Çfarë duhet të vëzhgojmë?** Pas ekzekutimit, kontrolloni jo vetëm vlerën numerike, por edhe nëse drejtimi i ndryshimit, shkalla e madhësive dhe forma e grafikut janë në përputhje me pritshmërinë teorike. Kjo e kthen qelizën nga një komandë programimi në një hap të arsyetimit shkencor.

In [ ]:
# Qëllimi i këtij blloku: Një dështim i qëllimshëm i k-means.
# Ngarkojmë vetëm mjetet që duhen për këtë hap të analizës.
from sklearn.datasets import make_moons
# Gjenerojmë të dhëna të kontrolluara që na lejojnë të dimë strukturën që duhet të zbulojë modeli.
Xm,_=make_moons(n_samples=450,noise=0.07,random_state=RANDOM_STATE)
# Përshtatim parametrat e modelit mbi bashkësinë e trajnimit.
km=KMeans(2,random_state=RANDOM_STATE,n_init=10).fit(Xm)
# Ndërtojmë një figurë diagnostike; grafiku është pjesë e analizës, jo dekorim.
fig,ax=plt.subplots()
ax.scatter(Xm[:,0],Xm[:,1],c=km.labels_,s=12)
ax.set_title("k-means imponon grupe afërsisht sferike")
plt.show()

**Pyetje për interpretim.** Çfarë përfundimi lejon ky rezultat dhe çfarë **nuk** lejon të përfundojmë? Nëse do të ndryshonim zhurmën, madhësinë e kampionit ose ndarjen e të dhënave, cilën pjesë të rezultatit prisni të ishte më e ndjeshme?

**Interpretim.** k-means optimizon një funksion të caktuar gjeometrik; ai nuk 'zbulon' domosdoshmërisht grupet që ne konsiderojmë fizikisht të kuptimshme.

Në një analizë reale ky interpretim duhet të shoqërohet me kontroll të njësive, madhësive karakteristike dhe hipotezave që lejojnë kalimin nga një tregues numerik te një përfundim fizik.

# 7. Aplikim jo-fizik: segmentim klientësh — EXTENDED

Në marketing, të njëjtat mjete përdoren për segmentim, por interpretimi i grupeve duhet të bëhet me kujdes: grupet algoritmike nuk janë kategori natyrore të garantuara.

# Ushtrime — PLAYGROUND

        ## A. Ushtrime të shkurtra
        1. Përsëritni k-means me 20 seed dhe analizoni ndryshueshmërinë.
2. Krahasoni k=2,3,4 me silhouette.
3. Ndryshoni `linkage` në dendrogram.
4. Shpjegoni pse scaling ndryshon grupimin.

        ## B. Ushtrim i plotë
        Gjeneroni tri faza fizike me forma jo-sferike dhe krahasoni k-means me AgglomerativeClustering.

        ## C. Playground
        Ndryshoni k, seed, zhurmën dhe standardizimin.

In [ ]:
# TODO / PLAYGROUND
# Ndryshoni parametrat, shtoni grafikë ose krahasoni modele të tjera.

## Përmbledhje konceptuale e kapitullit

Notebook-u **Grupimi dhe zbulimi i strukturës** duhet të lihet me një ide të qartë: algoritmi është vetëm një hallkë e arsyetimit. Para tij kemi zgjedhjen e përfaqësimit dhe hipotezat mbi të dhënat; pas tij kemi validimin, diagnostikën dhe interpretimin. Nëse një student mund të ekzekutojë të gjitha qelizat por nuk mund të shpjegojë pse secila prej tyre është e nevojshme, atëherë puna llogaritëse nuk është përfunduar ende.

Për studim të mëtejshëm, rikthehuni te shënimet e leksionit për derivimet e plota dhe përdoreni këtë notebook si laborator ku këto ide testohen numerikisht.